<a href="https://colab.research.google.com/github/karsarobert/LLM2026/blob/main/ChatGpt2025_10_feladatokkal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Chat GPT és más nagy nyelvi modellek alkalmazása
##PTE Gépi tanulás
###10. gyakorlat: RAG alapok – Beágyazások és szemantikus keresés
2026. április 13.

In [ ]:
from sentence_transformers import SentenceTransformer

# Modell betöltése – első futáskor letölti
modell = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print('Modell betöltve:', modell)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modell betöltve: SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


---
## 6. Szemantikus keresés egyszerű példán

Most összerakjuk az első **szemantikus keresőnket**!

A folyamat:
```
1. Dokumentumok beágyazása (egyszer)
2. Kérdés beágyazása
3. Hasonlóság számítása
4. Top-K leginkább hasonló dokumentum visszaadása
```

### 1. lépés – Mini tudásbázis létrehozása

10 mondat különböző témákról – ez lesz a "dokumentum kollekcióink".

In [ ]:
dokumentumok = [
    "A mesterséges intelligencia a számítástechnika egyik leggyorsabban fejlődő területe.",
    "A gépi tanulás segítségével a számítógépek adatokból tanulnak mintákat.",
    "A neurális hálózatok az emberi agy működését utánozzák.",
    "Python a legnépszerűbb programozási nyelv a data science területén.",
    "A Jupyter Notebook interaktív fejlesztői környezet Python kódhoz.",
    "A Pécs egy szép város Dél-Magyarországon, gazdag kulturális örökséggel.",
    "A PTE az ország egyik legrégebbi egyeteme, 1367-ben alapították.",
    "A magyarországi egyetemek közül a PTE kiemelkedik orvosképzésében.",
    "A kávé a világ egyik legnépszerűbb itala, reggel sokan fogyasztják.",
    "Az espresso egy tömény kávé típus, amelyet olasz módra készítenek."
]

print(f'{len(dokumentumok)} dokumentum betöltve')

10 dokumentum betöltve


### 2. lépés – Dokumentumok beágyazása

Ez a **lassú lépés** – de csak egyszer kell lefuttatni.
A beágyazásokat el lehet menteni és újra fel lehet használni.

In [ ]:
# Batch beágyazás – mind a 10 dokumentum egyszerre
dok_beagyazasok = modell.encode(dokumentumok, normalize_embeddings=True, show_progress_bar=True)

print(f'Beágyazások kész: {dok_beagyazasok.shape}')  # (10, 384)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Beágyazások kész: (10, 384)


### 3. lépés – Keresési függvény

Készítünk egy függvényt, amely:
1. Beágyazza a kérdést
2. Kiszámolja a hasonlóságokat
3. Visszaadja a top-K legjobb találatot

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def kereses(kerdes, top_k=3):
    """
    Szemantikus keresés a dokumentumok között.

    Args:
        kerdes: keresett szöveg
        top_k: hány találatot adjon vissza

    Returns:
        List of (index, pontszam, szoveg) tuple-ök
    """
    # Kérdés beágyazása
    kerdes_beagyazas = modell.encode(kerdes, normalize_embeddings=True)

    # Hasonlóságok számítása
    hasonlosagok = cosine_similarity([kerdes_beagyazas], dok_beagyazasok)[0]

    # Top-K index keresése
    top_indexek = np.argsort(hasonlosagok)[::-1][:top_k]  # csökkenő sorrend

    # Eredmények összeállítása
    eredmenyek = []
    for idx in top_indexek:
        eredmenyek.append((idx, hasonlosagok[idx], dokumentumok[idx]))

    return eredmenyek

print('Keresési függvény kész')

Keresési függvény kész


### 4. lépés – Keresési tesztek

Próbáljuk ki különböző kérdésekkel! Figyeljük meg, hogy a szemantikus keresés **nem kulcsszavakat** keres, hanem **jelentést**.

In [ ]:


# Tesztkérdések
print('=== Tesztkérdések ===')

# 1. keresés: AI témában
print('=== Keresés: "Mi az a gépi tanulás?" ===')
talalatok = kereses('Mi az a gépi tanulás?', top_k=3)

for i, (idx, pont, szoveg) in enumerate(talalatok, 1):
    print(f'{i}. [{pont:.4f}] {szoveg}')
print()

=== Tesztkérdések ===
=== Keresés: "Mi az a gépi tanulás?" ===
1. [0.5319] A gépi tanulás segítségével a számítógépek adatokból tanulnak mintákat.
2. [0.4626] A neurális hálózatok az emberi agy működését utánozzák.
3. [0.4219] Az espresso egy tömény kávé típus, amelyet olasz módra készítenek.



In [ ]:
# 2. keresés: Python témában
print('=== Keresés: "Milyen programozási nyelvet használjak adatelemzéshez?" ===')
talalatok = kereses('Milyen programozási nyelvet használjak adatelemzéshez?', top_k=3)

for i, (idx, pont, szoveg) in enumerate(talalatok, 1):
    print(f'{i}. [{pont:.4f}] {szoveg}')
print()

=== Keresés: "Milyen programozási nyelvet használjak adatelemzéshez?" ===
1. [0.5905] A kávé a világ egyik legnépszerűbb itala, reggel sokan fogyasztják.
2. [0.5598] A mesterséges intelligencia a számítástechnika egyik leggyorsabban fejlődő területe.
3. [0.5327] Az espresso egy tömény kávé típus, amelyet olasz módra készítenek.



In [ ]:
# 3. keresés: Pécs témában
print('=== Keresés: "Hol található a PTE?" ===')
talalatok = kereses('Hol található a PTE?', top_k=3)

for i, (idx, pont, szoveg) in enumerate(talalatok, 1):
    print(f'{i}. [{pont:.4f}] {szoveg}')
print()

=== Keresés: "Hol található a PTE?" ===
1. [0.4532] Az espresso egy tömény kávé típus, amelyet olasz módra készítenek.
2. [0.4328] A PTE az ország egyik legrégebbi egyeteme, 1367-ben alapították.
3. [0.3936] A magyarországi egyetemek közül a PTE kiemelkedik orvosképzésében.



In [ ]:
# 4. keresés: irreleváns kérdés
print('=== Keresés: "Hogyan készítsek pizzát?" ===')
talalatok = kereses('Hogyan készítsek pizzát?', top_k=3)

for i, (idx, pont, szoveg) in enumerate(talalatok, 1):
    print(f'{i}. [{pont:.4f}] {szoveg}')

print('\n→ Figyeljük meg: az irreleváns kérdésnél is vannak találatok,')
print('  de a pontszámok alacsonyak (általában < 0.3 = gyenge kapcsolat)')

=== Keresés: "Hogyan készítsek pizzát?" ===
1. [0.5559] A magyarországi egyetemek közül a PTE kiemelkedik orvosképzésében.
2. [0.5552] A kávé a világ egyik legnépszerűbb itala, reggel sokan fogyasztják.
3. [0.5436] Az espresso egy tömény kávé típus, amelyet olasz módra készítenek.

→ Figyeljük meg: az irreleváns kérdésnél is vannak találatok,
  de a pontszámok alacsonyak (általában < 0.3 = gyenge kapcsolat)


---
## 7. Dokumentum kollekció feldolgozása

Nagyobb dokumentum gyűjteményeknél a beágyazásokat érdemes **elmenteni**, hogy ne kelljen mindig újraszámolni őket.

### 1. lépés – Beágyazások mentése fájlba

NumPy `.npy` formátumban mentjük – gyors és hatékony.

In [ ]:
import numpy as np
import json

# Beágyazások mentése
np.save('dokumentum_beagyazasok.npy', dok_beagyazasok)

# Dokumentumok mentése JSON-ba (a szövegeket is el kell menteni!)
with open('dokumentumok.json', 'w', encoding='utf-8') as f:
    json.dump(dokumentumok, f, ensure_ascii=False, indent=2)

print('Beágyazások és dokumentumok elmentve')
print(f'  Beágyazások mérete: {dok_beagyazasok.nbytes / 1024:.1f} KB')

Beágyazások és dokumentumok elmentve
  Beágyazások mérete: 15.0 KB


### 2. lépés – Visszatöltés

A mentett beágyazások gyorsan betölthetők – nem kell újra futtatni a modellt.

In [ ]:
# Visszatöltés
betoltott_beagyazasok = np.load('dokumentum_beagyazasok.npy')

with open('dokumentumok.json', 'r', encoding='utf-8') as f:
    betoltott_dokumentumok = json.load(f)

print('Visszatöltve:')
print(f'  Beágyazások: {betoltott_beagyazasok.shape}')
print(f'  Dokumentumok: {len(betoltott_dokumentumok)} db')

# Ellenőrzés: ugyanazok?
print(f'\nAzonos: {np.allclose(dok_beagyazasok, betoltott_beagyazasok)}')

Visszatöltve:
  Beágyazások: (10, 384)
  Dokumentumok: 10 db

Azonos: True


### 3. lépés – Nagyobb dokumentum kollekció szimulálása

Készítsünk egy 50 dokumentumos adatbázist szakdolgozat témákról.

In [ ]:
# Témakörök bővítése
nagy_kollekcio = [
    # AI / ML
    "A mesterséges intelligencia a számítástechnika egyik leggyorsabban fejlődő területe.",
    "A gépi tanulás segítségével a számítógépek adatokból tanulnak mintákat.",
    "A neurális hálózatok az emberi agy működését utánozzák.",
    "A deep learning több rétegű neurális hálókat használ komplex minták felismerésére.",
    "A reinforcement learning játékokban és robotikában használatos.",
    "A természetes nyelvfeldolgozás (NLP) szövegek megértésével foglalkozik.",
    "A számítógépes látás képek és videók elemzésére szolgál.",
    "A GPT modellek transformer architektúrán alapulnak.",
    "Az embedding vektorok szemantikus információt kódolnak.",
    "A RAG rendszerek lekérdezéssel bővítik az LLM tudását.",

    # Programozás
    "Python a legnépszerűbb programozási nyelv a data science területén.",
    "A Jupyter Notebook interaktív fejlesztői környezet Python kódhoz.",
    "A pandas könyvtár táblázatos adatok kezelésére szolgál.",
    "A NumPy numerikus számításokhoz nyújt gyors tömbműveleteket.",
    "A scikit-learn klasszikus gépi tanulás algoritmusokat tartalmaz.",
    "A TensorFlow és PyTorch deep learning keretrendszerek.",
    "A Git verziókezelő rendszer a kódok nyomon követésére.",
    "A Docker konténerizációt biztosít alkalmazásokhoz.",
    "A FastAPI modern Python web API keretrendszer.",
    "A SQL adatbázis lekérdező nyelv relációs adatbázisokhoz.",

    # Egyetem / Pécs
    "A Pécs egy szép város Dél-Magyarországon, gazdag kulturális örökséggel.",
    "A PTE az ország egyik legrégebbi egyeteme, 1367-ben alapították.",
    "A magyarországi egyetemek közül a PTE kiemelkedik orvosképzésében.",
    "A Pécsi Tudományegyetem 10 karra tagozódik.",
    "A PTE Természettudományi Karán tanulható informatika.",
    "A pécsi egyetem nemzetközi programokat is kínál.",
    "A Zsolnay negyedben található a PTE BTK épülete.",
    "A PMMIK a Pollack Mihály Műszaki és Informatikai Kar.",
    "A Pécs Európa Kulturális Fővárosa volt 2010-ben.",
    "A pécsi Dzsámi a török hódoltság emléke.",

    # Adattudomány
    "Az adattudomány adatokból nyeri ki a hasznos tudást.",
    "A statisztika az adatelemzés alapja.",
    "A nagy adatok (Big Data) speciális feldolgozási módszereket igényelnek.",
    "Az ETL folyamat: Extract, Transform, Load.",
    "A feature engineering javítja a modellek teljesítményét.",
    "A cross-validation modellértékelési technika.",
    "Az overfitting esetén a modell túltanulja a tréning adatokat.",
    "A confusion matrix osztályozási eredményeket vizualizál.",
    "A ROC görbe a klasszifikátorok teljesítményét mutatja.",
    "A hiperparaméter hangolás javítja a modell pontosságát.",

    # Web / mobil
    "A React JavaScript könyvtár felhasználói felületek építéséhez.",
    "A REST API HTTP protokollon keresztül kommunikál.",
    "A JSON az API-k leggyakoribb adatformátuma.",
    "A Flutter platformfüggetlen mobil alkalmazások fejlesztésére szolgál.",
    "A HTML a weboldalak struktúráját definiálja.",
    "A CSS a weboldalak stílusáért felel.",
    "A JavaScript a weboldalak interaktivitását biztosítja.",
    "A Node.js szerver oldali JavaScript futtatókörnyezet.",
    "A MongoDB NoSQL dokumentum adatbázis.",
    "A GraphQL alternatíva a REST API-khoz.",
]

print(f'{len(nagy_kollekcio)} dokumentum betöltve')

50 dokumentum betöltve


### 4. lépés – Nagy kollekció beágyazása

Ez most már több időt vehet igénybe. A `show_progress_bar=True` mutatja a haladást.

In [ ]:
import time

print('Beágyazás indítása...')
start = time.time()

nagy_beagyazasok = modell.encode(nagy_kollekcio, normalize_embeddings=True, show_progress_bar=True)

elapsed = time.time() - start
print(f'\nKész: {nagy_beagyazasok.shape} – {elapsed:.2f} másodperc ({len(nagy_kollekcio)/elapsed:.1f} dok/sec)')

Beágyazás indítása...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Kész: (50, 384) – 0.06 másodperc (872.4 dok/sec)


### 5. lépés – Keresés a nagy kollekcióban

Frissítsük a keresési függvényt, hogy az új kollekcióban keressen.

In [ ]:
def nagy_kereses(kerdes, top_k=5):
    """Keresés a nagy dokumentum kollekcióban."""
    kerdes_beagyazas = modell.encode(kerdes, normalize_embeddings=True)
    hasonlosagok = cosine_similarity([kerdes_beagyazas], nagy_beagyazasok)[0]
    top_indexek = np.argsort(hasonlosagok)[::-1][:top_k]

    eredmenyek = []
    for idx in top_indexek:
        eredmenyek.append((idx, hasonlosagok[idx], nagy_kollekcio[idx]))
    return eredmenyek

# Teszt
print('=== Keresés: "transformer architektúra" ===')
for i, (idx, pont, szoveg) in enumerate(nagy_kereses('transformer architektúra', top_k=3), 1):
    print(f'{i}. [{pont:.4f}] {szoveg}')

=== Keresés: "transformer architektúra" ===
1. [0.7591] A GPT modellek transformer architektúrán alapulnak.
2. [0.3378] A Git verziókezelő rendszer a kódok nyomon követésére.
3. [0.3019] A Pécs Európa Kulturális Fővárosa volt 2010-ben.


---
## 8. RAG pipeline Groq API-val

Most összerakjuk a **teljes RAG pipeline-t**:

```
Kérdés
  ↓
Szemantikus keresés → Top-K dokumentum
  ↓
Kontextus + kérdés → Groq API (OpenAI-kompatibilis)
  ↓
Válasz
```

### 1. lépés – Groq API inicializálása (OpenAI-kompatibilis)

Használjuk a Groq API-t OpenAI-kompatibilis módon (ahogy a 8. gyakorlatban láttuk).

In [ ]:
!pip install openai --quiet

print('openai csomag telepítve')

openai csomag telepítve


In [ ]:
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get('GROQ_API_KEY')
    print('API kulcs betöltve (Colab titkos tárolóból)')
except Exception:
    import getpass
    api_key = getpass.getpass('Groq API kulcs: ')
    print('API kulcs betöltve (manuálisan)')

# OpenAI kliens létrehozása Groq-ra irányítva
client = OpenAI(
    base_url='https://api.groq.com/openai/v1',
    api_key=api_key
)

# Modell választása
MODELL = 'openai/gpt-oss-120b'

print(f'Groq API kész, modell: {MODELL}')

API kulcs betöltve (Colab titkos tárolóból)
Groq API kész, modell: openai/gpt-oss-120b


### 2. lépés – RAG válaszadó függvény

Ez a függvény:
1. Keres releváns dokumentumokat
2. Összeállítja a promptot (kontextus + kérdés)
3. Elküldi a Gemini-nek
4. Visszaadja a választ

In [ ]:
def rag_valasz(kerdes, top_k=3):
    """
    RAG alapú válaszadás: keresés + LLM generálás.

    Args:
        kerdes: a felhasználó kérdése
        top_k: hány dokumentumot használjon kontextusként

    Returns:
        (valasz, talalatok) tuple
    """
    # 1. Keresés
    talalatok = nagy_kereses(kerdes, top_k=top_k)

    # 2. Kontextus összeállítása
    kontextus = '\n\n'.join([f'[{i+1}] {szoveg}' for i, (_, _, szoveg) in enumerate(talalatok)])

    # 3. Prompt készítése
    prompt = f"""Az alábbi dokumentumok alapján válaszolj a kérdésre.
Ha a válasz nem található meg a dokumentumokban, mondd azt, hogy "Nem találtam releváns információt".

DOKUMENTUMOK:
{kontextus}

KÉRDÉS: {kerdes}

VÁLASZ:"""

    # 4. LLM hívás (Groq API, OpenAI-kompatibilis módon)
    valasz = client.chat.completions.create(
        model=MODELL,
        messages=[
            {'role': 'user', 'content': prompt}
        ]
    )

    return valasz.choices[0].message.content, talalatok

print('RAG függvény kész')

RAG függvény kész


### 3. lépés – RAG tesztelés

Teszteljük különböző kérdésekkel. A válasz mellett látjuk a felhasznált dokumentumokat is.

In [ ]:
# 1. RAG kérdés: MI témában
kerdes1 = "Mire szolgál a RAG rendszer?"

print(f'KÉRDÉS: {kerdes1}\n')
valasz1, talalatok1 = rag_valasz(kerdes1, top_k=3)

print('FELHASZNÁLT DOKUMENTUMOK:')
for i, (idx, pont, szoveg) in enumerate(talalatok1, 1):
    print(f'  {i}. [{pont:.3f}] {szoveg}')

print(f'\nVÁLASZ:\n{valasz1}')

KÉRDÉS: Mire szolgál a RAG rendszer?

FELHASZNÁLT DOKUMENTUMOK:
  1. [0.611] A RAG rendszerek lekérdezéssel bővítik az LLM tudását.
  2. [0.452] A számítógépes látás képek és videók elemzésére szolgál.
  3. [0.410] A természetes nyelvfeldolgozás (NLP) szövegek megértésével foglalkozik.

VÁLASZ:
A RAG (Retrieval‑Augmented Generation) rendszer a lekérdezés (információ‑keresés) segítségével bővíti a nagy nyelvi modellek (LLM) tudását.


In [ ]:
# 2. RAG kérdés: Python témában
kerdes2 = "Milyen Python könyvtárakat használjak adatelemzéshez?"

print(f'KÉRDÉS: {kerdes2}\n')
valasz2, talalatok2 = rag_valasz(kerdes2, top_k=5)

print('FELHASZNÁLT DOKUMENTUMOK:')
for i, (idx, pont, szoveg) in enumerate(talalatok2, 1):
    print(f'  {i}. [{pont:.3f}] {szoveg}')

print(f'\nVÁLASZ:\n{valasz2}')

KÉRDÉS: Milyen Python könyvtárakat használjak adatelemzéshez?

FELHASZNÁLT DOKUMENTUMOK:
  1. [0.617] A Jupyter Notebook interaktív fejlesztői környezet Python kódhoz.
  2. [0.567] A magyarországi egyetemek közül a PTE kiemelkedik orvosképzésében.
  3. [0.553] A pandas könyvtár táblázatos adatok kezelésére szolgál.
  4. [0.490] A NumPy numerikus számításokhoz nyújt gyors tömbműveleteket.
  5. [0.470] A Pécs egy szép város Dél-Magyarországon, gazdag kulturális örökséggel.

VÁLASZ:
Az adatelemzéshez a dokumentumokban említett legfontosabb Python könyvtárak:

- **pandas** – táblázatos adatok (DataFrame‑ek) kezelésére, tisztításra és elemzésre.  
- **NumPy** – numerikus számításokhoz, gyors tömbműveletekhez, amelyek a pandas mögött is működnek.  
- **Jupyter Notebook** – interaktív fejlesztői környezet, amelyben kényelmesen kombinálhatod a kódot, a dokumentációt és a vizualizációt az adatelemzés során.

Ezeket a könyvtárakat gyakran használják együtt az adatok importálásához, előfeldolgozá

In [ ]:
# 3. RAG kérdés: PTE témában
kerdes3 = "Mikor alapították a PTE-t és hol található?"

print(f'KÉRDÉS: {kerdes3}\n')
valasz3, talalatok3 = rag_valasz(kerdes3, top_k=3)

print('FELHASZNÁLT DOKUMENTUMOK:')
for i, (idx, pont, szoveg) in enumerate(talalatok3, 1):
    print(f'  {i}. [{pont:.3f}] {szoveg}')

print(f'\nVÁLASZ:\n{valasz3}')

KÉRDÉS: Mikor alapították a PTE-t és hol található?

FELHASZNÁLT DOKUMENTUMOK:
  1. [0.504] A PTE Természettudományi Karán tanulható informatika.
  2. [0.485] A Zsolnay negyedben található a PTE BTK épülete.
  3. [0.473] A PTE az ország egyik legrégebbi egyeteme, 1367-ben alapították.

VÁLASZ:
A PTE-t 1367-ben alapították, és a Zsolnay negyedben található.


In [ ]:
# 4. RAG kérdés: irreleváns téma (a dokumentumokban nincs)
kerdes4 = "Hogyan készítsek pizzát otthon?"

print(f'KÉRDÉS: {kerdes4}\n')
valasz4, talalatok4 = rag_valasz(kerdes4, top_k=3)

print('FELHASZNÁLT DOKUMENTUMOK:')
for i, (idx, pont, szoveg) in enumerate(talalatok4, 1):
    print(f'  {i}. [{pont:.3f}] {szoveg}')

print(f'\nVÁLASZ:\n{valasz4}')
print('\n→ Figyeljük meg: az LLM felismeri, hogy nincs releváns információ!')

KÉRDÉS: Hogyan készítsek pizzát otthon?

FELHASZNÁLT DOKUMENTUMOK:
  1. [0.562] A magyarországi egyetemek közül a PTE kiemelkedik orvosképzésében.
  2. [0.511] A PTE az ország egyik legrégebbi egyeteme, 1367-ben alapították.
  3. [0.501] A pécsi Dzsámi a török hódoltság emléke.

VÁLASZ:
Nem találtam releváns információt.

→ Figyeljük meg: az LLM felismeri, hogy nincs releváns információ!


### 4. lépés – RAG vs. LLM közvetlen összehasonlítás

Nézzük meg a különbséget: ugyanazt a kérdést feltesszük RAG-gel és anélkül is.

In [ ]:
teszt_kerdes = "Mikor alapították a Pécsi Tudományegyetemet?"

# LLM kontextus nélkül (közvetlen)
print('=== LLM válasz KONTEXTUS NÉLKÜL ===')
kozvetlen = client.chat.completions.create(
    model=MODELL,
    messages=[{'role': 'user', 'content': teszt_kerdes}]
)
print(kozvetlen.choices[0].message.content)
print()

# RAG válasz (kontextussal)
print('=== RAG válasz KONTEXTUSSAL ===')
rag_v, rag_t = rag_valasz(teszt_kerdes, top_k=2)
print(rag_v)
print()
print('Források:')
for i, (_, pont, szoveg) in enumerate(rag_t, 1):
    print(f'  {i}. {szoveg}')

=== LLM válasz KONTEXTUS NÉLKÜL ===
A Pécsi Tudományegyetem (röviden PTE) hivatalosan **1367-ben** kapta meg alapító okiratát, amikor a pécsi egyházmegyei tanács a Szent István-székesegyházhoz kapcsolódó felsőoktatási intézményt, a középiskolát és az egyetemet is megalapította. Bár a középkori egyetem csak rövid ideig működött, ez a dátum a Pécsi Tudományegyetem történelmi alapításának tekinthető. A modern egyetem, amely ma a természettudományokra, bölcsészettudományokra, egészségügyre és műszaki tudományokra is kiterjed, 1950-ben alakult újra, de alapító évként a 1367-es év az, amelyet a PTE hagyományosan idéz.

=== RAG válasz KONTEXTUSSAL ===
Nem találtam releváns információt.

Források:
  1. A Pécsi Tudományegyetem 10 karra tagozódik.
  2. A pécsi egyetem nemzetközi programokat is kínál.


In [ ]:
teszt_kerdes = "Mikor alapították a PTE-t?"

# LLM kontextus nélkül (közvetlen)
print('=== LLM válasz KONTEXTUS NÉLKÜL ===')
kozvetlen = client.chat.completions.create(
    model=MODELL,
    messages=[{'role': 'user', 'content': teszt_kerdes}]
)
print(kozvetlen.choices[0].message.content)
print()

# RAG válasz (kontextussal)
print('=== RAG válasz KONTEXTUSSAL ===')
rag_v, rag_t = rag_valasz(teszt_kerdes, top_k=2)
print(rag_v)
print()
print('Források:')
for i, (_, pont, szoveg) in enumerate(rag_t, 1):
    print(f'  {i}. {szoveg}')

=== LLM válasz KONTEXTUS NÉLKÜL ===
A PTE, vagyis a **Pécsi Tudományegyetem**, 1367 ben lett megalapítva. Az egyetemet II. Károly király adományozta a városnak, és azóta a magyar felsőoktatás egyik legrégebbi intézménye.

=== RAG válasz KONTEXTUSSAL ===
1367-ben alapították a PTE-t.

Források:
  1. A PTE az ország egyik legrégebbi egyeteme, 1367-ben alapították.
  2. A PTE Természettudományi Karán tanulható informatika.


---
## 9. Embedding modellek összehasonlítása

Különböző embedding modellek **más-más eredményt** adhatnak.
Hasonlítsuk össze néhány népszerű modellt!

### 1. lépés – Modellek listája

Választunk 3-4 különböző modellt méret és nyelvi támogatás szerint.

In [ ]:
# Modellek listája – név: (modell_azonosító, dimenzió, leírás)
osszehasonlitando_modellek = {
    'MiniLM-L6': ('sentence-transformers/all-MiniLM-L6-v2', 384, 'Gyors, kis méret, angol'),
    'MPNet': ('sentence-transformers/all-mpnet-base-v2', 768, 'Pontosabb, nagyobb, angol'),
    'Multilingual': ('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', 384, 'Többnyelvű (magyar is)'),
}

print('Összehasonlítandó modellek:')
for nev, (model_id, dim, leiras) in osszehasonlitando_modellek.items():
    print(f'  {nev}: {leiras} – {dim}D')

Összehasonlítandó modellek:
  MiniLM-L6: Gyors, kis méret, angol – 384D
  MPNet: Pontosabb, nagyobb, angol – 768D
  Multilingual: Többnyelvű (magyar is) – 384D


### 2. lépés – Teszt adatok

Egy kérdés és dokumentumok – látjuk majd, melyik modell találja meg a legjobb egyezést.

In [ ]:
teszt_kerdes_osszeh = "Mire használható a gépi tanulás?"

teszt_dokumentumok = [
    "A gépi tanulás segítségével a számítógépek adatokból tanulnak mintákat.",  # RELEVÁNS
    "A Python programozási nyelv népszerű.",  # IRRELEVÁNS
    "A neurális hálózatok az AI alapjai.",  # KÖZEPES
    "Az adatelemzés fontos az üzleti döntésekhez.",  # IRRELEVÁNS
    "A machine learning algoritmusok predikciót készítenek adatokból.",  # RELEVÁNS
]

print('Teszt kérdés:', teszt_kerdes_osszeh)
print(f'{len(teszt_dokumentumok)} teszt dokumentum')

Teszt kérdés: Mire használható a gépi tanulás?
5 teszt dokumentum


### 3. lépés – Minden modell tesztelése

Minden modellel beágyazzuk a kérdést és dokumentumokat, majd kiszámoljuk a hasonlóságokat.

In [ ]:
eredmenyek_modellek = {}

for nev, (model_id, dim, leiras) in osszehasonlitando_modellek.items():
    print(f'\n=== {nev} betöltése ===')

    # Modell betöltése
    m = SentenceTransformer(model_id)

    # Beágyazások
    q_emb = m.encode(teszt_kerdes_osszeh, normalize_embeddings=True)
    d_emb = m.encode(teszt_dokumentumok, normalize_embeddings=True)

    # Hasonlóságok
    hasonlosagok = cosine_similarity([q_emb], d_emb)[0]

    # Sorrend
    sorrend = np.argsort(hasonlosagok)[::-1]

    eredmenyek_modellek[nev] = {
        'hasonlosagok': hasonlosagok,
        'sorrend': sorrend
    }

    print(f'Top-3:')
    for i, idx in enumerate(sorrend[:3], 1):
        print(f'  {i}. [{hasonlosagok[idx]:.4f}] {teszt_dokumentumok[idx][:60]}...')


=== MiniLM-L6 betöltése ===


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Top-3:
  1. [0.6737] A gépi tanulás segítségével a számítógépek adatokból tanulna...
  2. [0.5218] Az adatelemzés fontos az üzleti döntésekhez....
  3. [0.4580] A neurális hálózatok az AI alapjai....

=== MPNet betöltése ===


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Top-3:
  1. [0.7948] A gépi tanulás segítségével a számítógépek adatokból tanulna...
  2. [0.7180] A machine learning algoritmusok predikciót készítenek adatok...
  3. [0.4424] A neurális hálózatok az AI alapjai....


### 4. lépés – Összehasonlító táblázat

Rakjuk táblázatba az eredményeket – látható lesz, melyik modell adta a legjobb sorrendet.

In [ ]:
import pandas as pd

# DataFrame készítése
osszehasonlitas_data = []

for i, dok in enumerate(teszt_dokumentumok):
    sor = {'Dokumentum': dok[:50] + '...'}
    for nev in eredmenyek_modellek.keys():
        sor[nev] = f"{eredmenyek_modellek[nev]['hasonlosagok'][i]:.3f}"
    osszehasonlitas_data.append(sor)

df_osszeh = pd.DataFrame(osszehasonlitas_data)
print(df_osszeh.to_string(index=False))

                                           Dokumentum MiniLM-L6 MPNet Multilingual
A gépi tanulás segítségével a számítógépek adatokb...     0.674 0.589        0.795
             A Python programozási nyelv népszerű....     0.256 0.349        0.202
               A neurális hálózatok az AI alapjai....     0.458 0.487        0.442
      Az adatelemzés fontos az üzleti döntésekhez....     0.522 0.555        0.335
A machine learning algoritmusok predikciót készíte...     0.305 0.211        0.718


### 5. lépés – Értékelés

Melyik modell találta meg a 0. és 4. dokumentumot (a két RELEVÁNS-t) a legmagasabb pontszámmal?

In [ ]:
print('=== Releváns dokumentumok (0. és 4. index) átlagos pontszáma ===')
print()

for nev, eredmeny in eredmenyek_modellek.items():
    h = eredmeny['hasonlosagok']
    relevans_atlag = (h[0] + h[4]) / 2  # 0. és 4. index átlaga
    print(f'{nev:15s}: {relevans_atlag:.4f}')

print()
print('→ Magasabb érték = jobb modell ennél a feladatnál')

=== Releváns dokumentumok (0. és 4. index) átlagos pontszáma ===

MiniLM-L6      : 0.4895
MPNet          : 0.4000
Multilingual   : 0.7564

→ Magasabb érték = jobb modell ennél a feladatnál


### Több nyelv összehasonlítása

Hasonlítsd össze, hogyan teljesít ugyanaz a kérdés angol és magyar nyelven!

**Lépések:**
1. Vedd az angol `all-MiniLM-L6-v2` és a többnyelvű `paraphrase-multilingual-MiniLM-L12-v2` modellt
2. Készíts egy mini dokumentum gyűjteményt **magyarul**
3. Készíts egy kérdést **angolul** és **magyarul** is
4. Teszteld mindkét modellt mindkét kérdéssel
5. Elemzd: melyik kombináció működik, melyik nem?

In [ ]:
# Modellek betöltése
modell_angol = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
modell_tobbnyelvű = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

print('Modellek betöltve')
print(f'  Angol dimenzió: {modell_angol.get_sentence_embedding_dimension()}')
print(f'  Többnyelvű dimenzió: {modell_tobbnyelvű.get_sentence_embedding_dimension()}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modellek betöltve
  Angol dimenzió: 384
  Többnyelvű dimenzió: 384


In [ ]:
# Magyar dokumentumok
magyar_dokumentumok = [
    "A gépi tanulás az adattudomány egyik legfontosabb ága.",
    "Python programozási nyelv az AI fejlesztéshez.",
    "A neurális hálózatok mély tanulást tesznek lehetővé.",
    "Az adatvizualizáció segít megérteni a mintákat.",
    "A természetes nyelvfeldolgozás szövegek elemzésével foglalkozik.",
]

# Kérdések
kerdes_magyar = "Mire használható a gépi tanulás?"
kerdes_angol = "What is machine learning used for?"

print('Dokumentumok és kérdések kész')

Dokumentumok és kérdések kész


In [ ]:
# Beágyazások – mindkét modellel
dok_angol = modell_angol.encode(magyar_dokumentumok, normalize_embeddings=True)
dok_tobbnyelvű = modell_tobbnyelvű.encode(magyar_dokumentumok, normalize_embeddings=True)

print('Dokumentumok beágyazva mindkét modellel')

Dokumentumok beágyazva mindkét modellel


In [ ]:
def nyelvi_teszt(modell, modell_nev, kerdes, dok_beagyazasok):
    """Egy modell + kérdés + dokumentum kombináció tesztelése."""
    q_emb = modell.encode(kerdes, normalize_embeddings=True)
    hasonlosagok = cosine_similarity([q_emb], dok_beagyazasok)[0]
    top_idx = np.argsort(hasonlosagok)[::-1][0]  # legjobb

    print(f'\n{modell_nev} | Kérdés: "{kerdes}"')
    print(f'  Legjobb találat [{hasonlosagok[top_idx]:.4f}]: {magyar_dokumentumok[top_idx]}')

    return hasonlosagok[top_idx]

# Mind a 4 kombináció
print('='*70)
print('ANGOL MODELL')
print('='*70)
p1 = nyelvi_teszt(modell_angol, 'Angol modell', kerdes_magyar, dok_angol)
p2 = nyelvi_teszt(modell_angol, 'Angol modell', kerdes_angol, dok_angol)

print('\n' + '='*70)
print('TÖBBNYELVŰ MODELL')
print('='*70)
p3 = nyelvi_teszt(modell_tobbnyelvű, 'Többnyelvű', kerdes_magyar, dok_tobbnyelvű)
p4 = nyelvi_teszt(modell_tobbnyelvű, 'Többnyelvű', kerdes_angol, dok_tobbnyelvű)

ANGOL MODELL

Angol modell | Kérdés: "Mire használható a gépi tanulás?"
  Legjobb találat [0.7047]: A gépi tanulás az adattudomány egyik legfontosabb ága.

Angol modell | Kérdés: "What is machine learning used for?"
  Legjobb találat [0.1393]: Python programozási nyelv az AI fejlesztéshez.

TÖBBNYELVŰ MODELL

Többnyelvű | Kérdés: "Mire használható a gépi tanulás?"
  Legjobb találat [0.7979]: A gépi tanulás az adattudomány egyik legfontosabb ága.

Többnyelvű | Kérdés: "What is machine learning used for?"
  Legjobb találat [0.7765]: A gépi tanulás az adattudomány egyik legfontosabb ága.


In [ ]:
# Összehasonlítás
print('\n' + '='*70)
print('ÖSSZEHASONLÍTÁS')
print('='*70)
print()
print(f'Angol modell + magyar kérdés:       {p1:.4f}')
print(f'Angol modell + angol kérdés:        {p2:.4f}  ← nem működik (magyar dok!)')
print(f'Többnyelvű modell + magyar kérdés:  {p3:.4f}')
print(f'Többnyelvű modell + angol kérdés:   {p4:.4f}  ← működik!')
print()
print('KÖVETKEZTETÉS:')
print('- Az angol modell NEM érti az angol kérdést, ha a dokumentumok magyarok')
print('- A többnyelvű modell mindkét nyelven jól működik')
print('- Cross-lingual kereséshez (más nyelv kérdés vs. dokumentum) többnyelvű modell kell!')


ÖSSZEHASONLÍTÁS

Angol modell + magyar kérdés:       0.7047
Angol modell + angol kérdés:        0.1393  ← nem működik (magyar dok!)
Többnyelvű modell + magyar kérdés:  0.7979
Többnyelvű modell + angol kérdés:   0.7765  ← működik!

KÖVETKEZTETÉS:
- Az angol modell NEM érti az angol kérdést, ha a dokumentumok magyarok
- A többnyelvű modell mindkét nyelven jól működik
- Cross-lingual kereséshez (más nyelv kérdés vs. dokumentum) többnyelvű modell kell!


---
## 10. Feladatok

### 10.1 Feladat – Szakdolgozat címek keresése

Készíts egy mini adatbázist 15 szakdolgozat címből (5-5 AI, web fejlesztés, adattudomány témában).

**Lépések:**
1. Definiáld a 15 címet egy listában
2. Ágyazd be őket a modellel
3. Írj egy keresési függvényt, amely témakör alapján ad vissza top-3 címet
4. Teszteld: "mesterséges intelligencia alkalmazások", "frontend fejlesztés", "big data elemzés"

In [ ]:
modell = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

print('Modell betöltve:', modell)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Modell betöltve: SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)


In [ ]:
szakdolgozat_cimek = [
    # AI témák
    "Chatbot fejlesztése természetes nyelvfeldolgozással",
    "Képfelismerő rendszer mély tanulással",
    "Gépi tanulás alapú ajánlórendszer implementálása",
    "Hangfelismerés neurális hálózatokkal",
    "Automatikus szövegösszefoglaló LLM-mel",

    # Web fejlesztés
    "E-kereskedelmi platform React és Node.js alapokon",
    "Reszponzív webdesign modern CSS keretrendszerekkel",
    "REST API fejlesztés FastAPI használatával",
    "Valós idejű chat alkalmazás WebSocket technológiával",
    "Progressive Web App készítése Vue.js-sel",

    # Adattudomány
    "Nagy adathalmazok feldolgozása Apache Spark-kal",
    "Prediktív analitika idősor adatokon",
    "Adatvizualizációs dashboard Python Dash-sel",
    "Ügyfélszegmentáció klaszterezési algoritmusokkal",
    "ETL pipeline építése Python és SQL használatával",
]

print(f'{len(szakdolgozat_cimek)} szakdolgozat cím betöltve')

15 szakdolgozat cím betöltve


In [ ]:
# Beágyazás
szakdoga_beagyazasok =


Beágyazások: (15, 384)


In [ ]:
#

### 10.2 Feladat – FAQ rendszer RAG-gel

Készíts egy FAQ (gyakran ismételt kérdések) rendszert RAG alapon!

**Lépések:**
1. Definiálj 8-10 FAQ párt (kérdés + válasz) egy témában (pl. egyetemi beiratkozás, könyvtárhasználat)
2. A válaszokat ágyazd be
3. Készíts RAG függvényt, amely:
   - Keres a beágyazott válaszok között
   - A top-2 találatot átadja a Gemini-nek kontextusként
   - A Gemini természetes nyelven válaszol
4. Teszteld 3 különböző kérdéssel

In [ ]:
# FAQ adatok – egyetemi könyvtár téma
faq_adatok = [
    {
        "kerdes": "Hogyan iratkozhatok be a könyvtárba?",
        "valasz": "A beiratkozáshoz hozd magaddal a diákigazolványodat és egy fotót. A beiratkozás ingyenes PTE hallgatóknak."
    },
    {
        "kerdes": "Meddig kölcsönözhetek könyveket?",
        "valasz": "A kölcsönzési időszak 30 nap, ami egyszer meghosszabbítható újabb 30 nappal, ha senki más nem foglalta le a könyvet."
    },
    {
        "kerdes": "Milyen büntetés jár a késedelmes visszavitelért?",
        "valasz": "A késedelmi díj naponta 50 Ft könyvenként. 14 nap késedelem után a beiratkozás felfüggesztésre kerül."
    },
    {
        "kerdes": "Hány könyvet kölcsönözhetek egyszerre?",
        "valasz": "Alapértelmezetten 5 könyvet kölcsönözhetsz egyszerre. PhD hallgatók esetében ez a limit 10."
    },
    {
        "kerdes": "Van-e lehetőség online kölcsönzésre?",
        "valasz": "Igen, a könyvtár katalógusában online is le tudod foglalni a könyveket, és értesítést kapsz, amikor elérhetők."
    },
    {
        "kerdes": "Milyen nyitvatartási ideje van a könyvtárnak?",
        "valasz": "Hétfőtől péntekig 8:00-20:00 óráig, szombaton 9:00-14:00 óráig. Vasárnap zárva tartunk."
    },
    {
        "kerdes": "Lehet-e tanulóhelyet foglalni?",
        "valasz": "Igen, a központi könyvtárban online rendszeren keresztül foglalhatsz tanulóhelyet maximum 3 órára."
    },
    {
        "kerdes": "Hol találom a szakdolgozatokat?",
        "valasz": "A szakdolgozatok a kari könyvtárak zárt anyagában találhatók. Kérésre helyben olvashatók, de nem kölcsönözhetők."
    },
]

print(f'{len(faq_adatok)} FAQ bejegyzés betöltve')

8 FAQ bejegyzés betöltve


In [ ]:
# Csak a válaszokat ágyazzuk be (ezekben keresünk)
faq_valaszok = [item['valasz'] for item in faq_adatok]
faq_beagyazasok =


FAQ beágyazások: (8, 384)


FAQ RAG függvény kész


In [ ]:
# Teszt 1
print('=== KÉRDÉS: "Mennyi időre vehetem ki a könyveket?" ===\n')


=== KÉRDÉS: "Mennyi időre vehetem ki a könyveket?" ===



In [ ]:
# Teszt 2
print('=== KÉRDÉS: "Mi a büntetés ha elkések a könyv visszavitelével?" ===\n')

=== KÉRDÉS: "Mi a büntetés ha elkések a könyv visszavitelével?" ===



In [ ]:
# Teszt 3 – irreleváns kérdés
print('=== KÉRDÉS: "Milyen étterem van a campus közelében?" ===\n')

=== KÉRDÉS: "Milyen étterem van a campus közelében?" ===



---
## Összefoglalás

| Fejezet | Téma | Kulcstechnika |
|---|---|---|
| 1. | Mi a RAG? | Lekérdezés + generálás kombinálása |
| 2. | Beágyazások | Szöveg → számvektor, szemantikus jelentés |
| 3. | Sentence Transformers | `SentenceTransformer()` – Hugging Face |
| 4. | Első beágyazás | `.encode()`, normalizálás |
| 5. | Hasonlóság | `cosine_similarity()` – vektorok közötti hasonlóság |
| 6. | Szemantikus keresés | Kérdés beágyazása + top-K dokumentum |
| 7. | Kollekció feldolgozás | `numpy.save()` – beágyazások mentése |
| 8. | RAG pipeline | Keresés → kontextus → LLM (Groq, OpenAI-kompatibilis) |
| 9. | Modellek összehasonlítás | Különböző embedding modellek tesztelése |

### A RAG folyamat lépései

```
1. ELŐKÉSZÍTÉS (egyszer)
   Dokumentumok → embedding modell → vektorok → mentés

2. KÉRDÉS ÉRKEZIK
   Kérdés → embedding modell → kérdés vektor

3. KERESÉS
   Cosine similarity (kérdés vektor, dokumentum vektorok)
   → Top-K leginkább hasonló dokumentum

4. GENERÁLÁS
   Prompt = kontextus (Top-K dok) + kérdés
   → LLM (Groq API)
   → Válasz
```

### Embedding modellek választása

| Modell | Dimenzió | Nyelv | Mire jó? |
|---|---|---|---|
| `all-MiniLM-L6-v2` | 384 | Angol | Gyors, kisméretű, általános |
| `all-mpnet-base-v2` | 768 | Angol | Pontosabb, de lassabb |
| `paraphrase-multilingual-*` | 384-768 | 50+ nyelv | Többnyelvű, cross-lingual |
| `multilingual-e5-*` | 384-1024 | 100+ nyelv | Nagy teljesítmény, többnyelvű |

### Mikor használj RAG-et?

✅ **Használd RAG-et, ha:**
- Friss adatokra van szükség (hírek, árak, készlet)
- Privát dokumentumokat kell feldolgozni (céges tudásbázis)
- Pontos forrásokra van szükség ("melyik dokumentumból jött a válasz?")
- Az adatok gyakran változnak

❌ **NE használd RAG-et, ha:**
- Az LLM alapértelmezett tudása elég (általános kérdések)
- Kreatív generálásra van szükség (creative writing)
- Nincsenek strukturált dokumentumaid

### API összehasonlítás

| Szolgáltató | Típus | Használt modell |
|---|---|---|
| **Gemini** | Google natív API | `gemini-1.5-flash` |
| **Groq** | OpenAI-kompatibilis | `llama-3.3-70b-versatile` |
| **OpenRouter** | OpenAI-kompatibilis | Különböző modellek |

**Előny:** Az OpenAI-kompatibilis API-k ugyanazzal a kóddal működnek – csak a `base_url` és `api_key` változik!

**Következő óra:** Haladó RAG technikák – chunk stratégiák, re-ranking, hibrid keresés, vector adatbázisok.